# Road Following Live (JetRacer ONNX + ROS Subscriber Pipeline)

This notebook subscribes to the **ROS Camera Topic** (`/csi_cam_0/image_raw`), executing ONNX model inference and **Stanley Control** on physical JetRacer.
It displays the live camera stream with prediction overlay directly under the notebook cell (without `ipywidgets` dependencies) and records the driving video to `output_drive.mp4`.

### 1. Setup Environment & Load ONNX Model

In [ ]:
import os
import sys
from pathlib import Path

# Add parent directory to sys.path to access Controller.py, Runner.py, and utils.py
parent_dir = Path.cwd().parent
if str(parent_dir) not in sys.path:
    sys.path.append(str(parent_dir))

import onnxruntime as ort
try:
    from jetracer.utils import preprocess_onnx, bgr8_to_jpeg
except ImportError:
    from utils import preprocess_onnx, bgr8_to_jpeg

# Locate ONNX model file
model_path = os.path.join(Path.cwd(), "road_following_model.onnx")
if not os.path.exists(model_path):
    model_path = os.path.join(parent_dir, "notebooks", "road_following_model.onnx")

if not os.path.exists(model_path):
    print(f"[!] ERROR: ONNX model file '{model_path}' not found!")
else:
    print(f"[*] Loading ONNX model from: {model_path}")

available_providers = ort.get_available_providers()
providers = ['CUDAExecutionProvider'] if 'CUDAExecutionProvider' in available_providers else []
providers.append('CPUExecutionProvider')

try:
    session = ort.InferenceSession(model_path, providers=providers)
except Exception:
    session = ort.InferenceSession(model_path, providers=['CPUExecutionProvider'])

input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name
print(f"[+] Loaded ONNX Session with providers: {session.get_providers()}")


### 2. Initialize ROS Node & JetRacer Hardware (`NvidiaRacecar`)

In [ ]:
import rospy
from sensor_msgs.msg import Image as ROSImage
from jetracer.nvidia_racecar import NvidiaRacecar
try:
    from jetracer.Controller import StanleyController
    from jetracer.Runner import JetRacerROSOnnxRunner
except ImportError:
    from Controller import StanleyController
    from Runner import JetRacerROSOnnxRunner

# 1. Initialize ROS Node
try:
    rospy.init_node('road_following_live_notebook', anonymous=True, disable_signals=True)
    print("[+] ROS Node initialized successfully!")
except Exception as e:
    print(f"[*] ROS Node notice: {e}")

# 2. Hardware & Controller Setup
car = NvidiaRacecar()
stanley = StanleyController()
stanley.reset()
print("[+] JetRacer hardware and Stanley Controller initialized.")


### 3. Setup In-Place HTML Live Display & ROS Subscriber

In [ ]:
import cv2
import base64
from IPython.display import display, HTML

# Create IPython Display Handle for zero-flicker HTML image streaming
dh = display(HTML("<p><b>Waiting for ROS Camera Feed...</b></p>"), display_id=True)

video_output_path = os.path.join(Path.cwd(), "output_drive.mp4")

def on_ros_frame(cv_image, raw_x, raw_y, smoothed_x, steering, dyn_throttle):
    h, w = cv_image.shape[:2]
    px = int(w * (smoothed_x / 2.0 + 0.5))
    py = int(h * (raw_y / 2.0 + 0.5)) if raw_y != 0.0 else int(h * 0.5)

    prediction = cv_image.copy()
    cv2.circle(prediction, (px, py), 8, (0, 255, 0), -1)
    cv2.putText(prediction, f"Steer:{steering:+.2f} Thr:{dyn_throttle:.2f}", (10, 25),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    jpeg_bytes = bgr8_to_jpeg(prediction)
    b64 = base64.b64encode(jpeg_bytes).decode('utf-8')
    
    html_content = f'''
    <div style="font-family: monospace; background: #1e1e1e; color: #00ff00; padding: 12px; border-radius: 8px; display: inline-block;">
        <h4 style="margin:0 0 8px 0; color: #ffffff;">JetRacer Autonomous Live Stream (ONNX + Stanley)</h4>
        <p style="margin:4px 0;"><b>Target X:</b> {raw_x:+.3f} | <b>Smoothed X:</b> {smoothed_x:+.3f}</p>
        <p style="margin:4px 0;"><b>Steering:</b> {steering:+.3f} | <b>Throttle:</b> {dyn_throttle:.3f}</p>
        <img src="data:image/jpeg;base64,{b64}" style="width:320px; height:auto; border:2px solid #00ff00; border-radius:4px; margin-top:8px;" />
    </div>
    '''
    dh.update(HTML(html_content))

runner = JetRacerROSOnnxRunner(
    session=session,
    input_name=input_name,
    output_name=output_name,
    car=car,
    stanley=stanley,
    k=2.5,
    throttle=0.20,
    brake_gain=0.10,
    bias=0.0,
    alpha=0.4,
    video_path=video_output_path,
    video_fps=20.0,
    on_frame=on_ros_frame
)

runner.running = False  # Start paused

topic_name = "/csi_cam_0/image_raw"
ros_sub = rospy.Subscriber(topic_name, ROSImage, runner.image_callback, queue_size=1, buff_size=2**24)
print(f"[*] Subscribed to ROS Image Topic: {topic_name}")


### 4. Run Autonomous Driving (Live Display below Cell + Video Recording)

In [ ]:
# Run this cell to START autonomous driving & see live camera feed right below!
# Press Stop/Interrupt kernel button to stop car safely.

import time

runner.running = True
stanley.reset()
print("[+] AUTONOMOUS DRIVING ACTIVE (Live Display updating below...)")

try:
    while runner.running:
        time.sleep(0.1)
except KeyboardInterrupt:
    pass
finally:
    runner.stop()


### 5. Emergency Stop Cell

In [ ]:
# Emergency Stop Cell
runner.stop()
